In [2]:
%pip install pandas numpy matplotlib seaborn scikit-learn lightgbm shap

Note: you may need to restart the kernel to use updated packages.


# Predicción de cancelación de reservas hoteleras

## Contexto de negocio
El equipo de Revenue Management de una cadena con dos hoteles en Portugal (City Hotel y Resort Hotel) necesita identificar, **en el momento de la reserva**, qué reservas tienen alta probabilidad de cancelarse.

Con esa predicción podrá activar acciones preventivas:
- Ofrecer upgrades con descuento.
- Solicitar depósito no reembolsable de forma selectiva.
- Ajustar la política de overbooking.

## Los dos errores posibles
| Error | Significado | Coste de negocio |
|---|---|---|
| Falso negativo | Cancelación no detectada | Habitación vacía de última hora, ingreso perdido |
| Falso positivo | Acción sobre un cliente que sí iba a venir | Molestia al cliente, posible pérdida de fidelidad |

El modelo debe equilibrar ambos errores.

## Datos
Hotel Booking Demand, de Antonio, Almeida y Nunes (2019). Contiene unas 119.000 reservas con llegada entre julio de 2015 y agosto de 2017.
Variable objetivo: `is_canceled` (1 = cancelada).

In [3]:
# === LIBRERÍAS ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# === CONFIGURACIÓN GENERAL === 
RANDOM_STATE = 42 # Semilla fija: garantiza resultados reproducibles en cada ejecución

pd.set_option("display.max_columns", None) # Mostrar todas las columnas (tenemos 32)
sns.set_theme(style="whitegrid") # Estilo limpio para los gráficos del EDA
plt.rcParams["figure.figsize"] = (10, 5) # Tamaño por defecto de las figuras

In [4]:
# Carga del dataset (el CSV está en la misma carpeta que el notebook)
df = pd.read_csv("reservas_hoteleras.csv")

print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
df.head()

Filas: 119,390 | Columnas: 32


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [5]:
# Tipo de dato de cada columna
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

In [6]:
# Valores nulos: cantidad y porcentaje, solo en las columnas que tienen alguno
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)

resumen_nulos = pd.DataFrame({"nulos": nulos, "porcentaje": nulos_pct})
resumen_nulos[resumen_nulos["nulos"] > 0].sort_values("nulos", ascending=False)

,nulos,porcentaje
company,112593,94.31
agent,16340,13.69
country,488,0.41
children,4,0.00


In [7]:
# Filas completamente identicas en las 32 filas
n_duplicados = df.duplicated().sum()
print(f"Filas duplicadas: {n_duplicados:,} ({n_duplicados / len(df):.1%})")

Filas duplicadas: 31,994 (26.8%)


In [8]:
# Distribución de target
print(df["is_canceled"].value_counts())
print()
print(df["is_canceled"].value_counts(normalize=True).round(3))

is_canceled
0    75166
1    44224
Name: count, dtype: int64

is_canceled
0    0.63
1    0.37
Name: proportion, dtype: float64


In [9]:
# Rango temporal de las llegadas
print(df.groupby("arrival_date_year")["arrival_date_month"].unique())

arrival_date_year
2015    [July, August, September, October, November, D...
2016    [January, February, March, April, May, June, J...
2017    [January, February, March, April, May, June, J...
Name: arrival_date_month, dtype: object


In [10]:
# Comprobación previa: ¿existe ya el valor 0 como ID de agencia o empresa?
print("agent == 0", (df["agent"] == 0).sum())
print("company == 0", (df["company"] == 0).sum())

agent == 0 0
company == 0 0


In [11]:
# === TRATAMIENTO DE NULOS === 
df_clean = df.copy() # Trabajaremos sobre una copia, así el df original queda intacto comparar

# agent / company: el nulo significa "reserva sin agencia" / "sin empresa" (nulo informativo)
df_clean["agent"] = df_clean["agent"].fillna(0).astype(int)
df_clean["company"] = df_clean["company"].fillna(0).astype(int)

# country: país desconocido -> categoría propia, para no inventar un país
df_clean["country"] = df_clean["country"].fillna("Unknown")

# children: solo 4 nulos, casi seguro un error de registro -> 0 (valor más frecuente)
df_clean["children"] = df_clean["children"].fillna(0).astype(int)

print("Nulos restantes:", df_clean.isnull().sum().sum())

Nulos restantes: 0


In [12]:
# Marcamos TODAS las filas que forman parte de un grupo de duplicados (keep=False)
es_dup = df_clean.duplicated(keep=False)

comparacion = pd.DataFrame({
    "tasa_cancelacion": [df_clean.loc[~es_dup, "is_canceled"].mean(),
                         df_clean.loc[es_dup, "is_canceled"].mean()],
    "n_filas": [(~es_dup).sum(), es_dup.sum()]
}, index=["únicas", "duplicadas"])
print(comparacion.round(3), "\n")

# ¿En qué segmento de mercado y tipo de depósito se concentran los dupicados?
print(df_clean.loc[es_dup, "market_segment"].value_counts(normalize=True).round(3), "\n")
print(df_clean.loc[es_dup, "deposit_type"].value_counts(normalize=True).round(3))

            tasa_cancelacion  n_filas
únicas                 0.262    79225
duplicadas             0.584    40165 

market_segment
Groups           0.416
Offline TA/TO    0.304
Online TA        0.206
Corporate        0.037
Direct           0.035
Complementary    0.002
Aviation         0.000
Name: proportion, dtype: float64 

deposit_type
No Deposit    0.642
Non Refund    0.356
Refundable    0.002
Name: proportion, dtype: float64


In [13]:
# ADR (precio medio por noche)
print(df_clean["adr"].describe().round(2), "\n")
print("ADR < 0:", (df_clean["adr"] < 0).sum())
print("ADR == 0:", (df_clean["adr"] == 0).sum())
print("ADR > 1000:", (df_clean["adr"] > 1000).sum(), "\n")

# ¿Qué segmentos tienen ADR = 0? (esperamos Complementary)
print(df_clean.loc[df_clean["adr"] == 0, "market_segment"].value_counts(), "\n")

# Reservas sin huespedes o sin noches
total_huespedes = df_clean["adults"] + df_clean["children"] + df_clean["babies"]
total_noches = df_clean["stays_in_weekend_nights"] + df_clean["stays_in_week_nights"]
print("Reservas con 0 huéspedes:", (total_huespedes == 0).sum())
print("reservas con 0 noches:", (total_noches == 0).sum())

count    119390.00
mean        101.83
std          50.54
min          -6.38
25%          69.29
50%          94.58
75%         126.00
max        5400.00
Name: adr, dtype: float64 

ADR < 0: 1
ADR == 0: 1959
ADR > 1000: 1 

market_segment
Complementary    680
Online TA        367
Offline TA/TO    332
Groups           252
Direct           240
Corporate         82
Aviation           6
Name: count, dtype: int64 

Reservas con 0 huéspedes: 180
reservas con 0 noches: 715


In [14]:
# === ELIMINACIÓN DE REGISTROS INVÁLIDOS ===
n_inicial = len(df_clean)

# ADR negativo o extremo (> 1000 €/noche): errores de registro, no precios reales
df_clean = df_clean[(df_clean["adr"] >= 0) & (df_clean["adr"] < 1000)]

# Reservas sin huespedes: no representan una reserva real
df_clean = df_clean[(df_clean["adults"] + df_clean["children"] + df_clean["babies"]) > 0]

print(f"Filas eliminadas: {n_inicial - len(df_clean)} ({(n_inicial - len(df_clean)) / n_inicial:.2%})")

# Los duplicados se MANTIENEN: son probablemente bloques de habitaciones de grupos/TTOO
# La partición temporal evita que copias idénticas acaben en train y test

# === Fecha de llegada completa ===
meses = {"January": 1, "February": 2, "March": 3, "April": 4, "May": 5, "June": 6, 
         "July": 7, "August": 8, "September": 9, "October": 10, "November": 11, "December": 12}

df_clean["arrival_month_num"] = df_clean["arrival_date_month"].map(meses)
df_clean["arrival_date"] = pd.to_datetime(dict(year=df_clean["arrival_date_year"],
                                               month=df_clean["arrival_month_num"],
                                               day=df_clean["arrival_date_day_of_month"]))

df_clean = df_clean.reset_index(drop=True)

print(f"Filas finales: {len(df_clean):,}")
print(f"Tasa de cancelación tras limpieza: {df_clean['is_canceled'].mean():.3f}")
print(f"Rango de llegadas: {df_clean['arrival_date'].min().date()} → {df_clean['arrival_date'].max().date()}")

Filas eliminadas: 182 (0.15%)
Filas finales: 119,208
Tasa de cancelación tras limpieza: 0.371
Rango de llegadas: 2015-07-01 → 2017-08-31
